# 01 · Preço da soja

Nesta etapa eu baixo o preço mensal da soja, calculo o que quero prever (o alvo) e as primeiras features, que usam só o próprio preço.

O preço é o do FMI, em dólares por tonelada (série `PSOYBUSDM` do FRED). Ele é a média do mês, então cada mês tem um único valor.

In [ ]:
import sys
from pathlib import Path

# No Kaggle o projeto está dentro de /kaggle/input; no computador, é a pasta acima de notebooks/
if Path("/kaggle/input").exists():
    RAIZ = next(Path("/kaggle/input").rglob("src/soja/__init__.py")).parents[2]
else:
    RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from soja import config, dados, features, avaliacao

ETAPA = "01_preco"
PASTA = config.pasta_resultados(ETAPA)
plt.rcParams.update({"figure.figsize": (11, 3.5), "axes.grid": True, "grid.alpha": .3})
print("Projeto em", RAIZ, "| resultados em", PASTA)

## 1. Baixando o preço

Se der erro de conexão, é só rodar a célula de novo.

In [ ]:
preco_bruto = dados.baixar_preco()
preco = preco_bruto.loc[config.INICIO:]
print(f"{len(preco)} meses, de {preco.index[0]} a {preco.index[-1]}")

preco.plot(color="tab:green", title="Preço da soja (US$/t)")
plt.axvspan(pd.Period(config.TESTE_INICIO), pd.Period(config.TESTE_FIM), color="gray", alpha=.15, label="período de teste")
plt.legend(); plt.show()
preco.tail()

## 2. Por que prever a variação e não o preço

O preço muda muito de patamar ao longo dos anos (de menos de 200 para mais de 600 US$/t). Se eu tentasse prever o preço direto, qualquer modelo que só copiasse o mês anterior já pareceria ótimo, porque o preço de um mês é quase igual ao do outro.

Por isso eu trabalho com a variação log, `ln P(t) − ln P(t−1)`, que é praticamente a variação percentual do mês. Ela fica em torno de zero o tempo todo e mostra o que realmente importa: quanto o preço subiu ou caiu.

In [ ]:
variacao = np.log(preco).diff()

fig, ax = plt.subplots(1, 2, figsize=(13, 3.5), gridspec_kw={"width_ratios": [3, 1]})
cores = np.where(variacao >= 0, "tab:green", "tab:red")
ax[0].bar(variacao.index.to_timestamp(), 100 * variacao, width=25, color=cores)
ax[0].set_title("Variação do preço no mês (%)")
ax[1].hist(100 * variacao.dropna(), bins=40, color="tab:blue")
ax[1].set_title("Distribuição da variação (%)")
fig.tight_layout(); fig.savefig(PASTA / "variacao.png", dpi=120); plt.show()

print(f"Variação média: {100 * variacao.mean():+.2f}% | desvio padrão: {100 * variacao.std():.2f}%")
print("Maiores altas e quedas:")
pd.concat([variacao.nlargest(3), variacao.nsmallest(3)]).mul(100).round(1)

## 3. O alvo e as features de preço

O **alvo** da linha do mês *t* é a variação do mês seguinte, `ln P(t+1) − ln P(t)`. A última linha fica sem alvo, porque é justamente o mês que eu quero prever.

As features de preço só usam o que já se sabia no fim do mês *t*:

| Feature | O que é |
|---|---|
| `dlog_preco` | quanto o preço variou neste mês |
| `dlog_preco_lag1` | quanto variou no mês anterior |
| `tendencia_3m` | quanto variou nos últimos 3 meses |

A ideia é ver se o preço tem algum "embalo": se depois de subir ele tende a continuar subindo.

In [ ]:
tabela = pd.concat([preco, features.features_preco(preco), features.alvo(preco)], axis=1)

# conferência: o alvo de um mês tem que ser igual à variação do mês seguinte
assert np.allclose(tabela["alvo"].iloc[:-1].values, tabela["dlog_preco"].iloc[1:].values)

# correlação de cada feature com o alvo, olhando só os anos antes do teste
treino = tabela.loc[:pd.Period(config.TESTE_INICIO) - 1]
corr = treino.drop(columns=["preco_usd_t", "alvo"]).corrwith(treino["alvo"])
corr.plot.bar(color="tab:blue", rot=0, title=f"Correlação com o alvo ({treino.index[0]} a {treino.index[-1]})")
plt.axhline(0, color="k", lw=.8); plt.show()

tabela.tail()

## 4. A referência: o passeio aleatório

O jeito mais simples de prever é dizer que o preço do mês que vem vai ser igual ao de agora, ou seja, que a variação vai ser zero. Isso é o passeio aleatório. O erro dele é o próprio alvo, então o RMSE dele é o tamanho típico da variação mensal.

Todo modelo que eu fizer daqui para frente vai ser comparado com esse número. Se não ganhar dele, não serve.

In [ ]:
teste = tabela.loc[config.TESTE_INICIO:config.TESTE_FIM, "alvo"]
rmse_passeio = avaliacao.rmse(teste)
print(f"Teste: {teste.index[0]} a {teste.index[-1]} ({len(teste)} meses)")
print(f"RMSE do passeio aleatório: {rmse_passeio:.4f} (cerca de {100 * rmse_passeio:.1f}% por mês)")

por_ano = teste.groupby(teste.index.year).apply(avaliacao.rmse)
por_ano.mul(100).plot.bar(color="tab:gray", rot=0, title="Erro do passeio aleatório por ano (RMSE, %)")
plt.axhline(100 * rmse_passeio, color="k", ls="--", lw=1, label="todo o teste"); plt.legend(); plt.show()

## 5. Salvando

Gravo em `results/01_preco/`:
- `preco.csv`: preço, features de preço e alvo, um mês por linha
- `config.json`: parâmetros da rodada
- `variacao.png`: figura da variação mensal

In [ ]:
tabela.index = tabela.index.astype(str)
tabela.index.name = "mes"
tabela.to_csv(PASTA / "preco.csv")
config.salvar_config(
    PASTA, fonte="FRED PSOYBUSDM (FMI, US$/t)", meses=len(tabela),
    ultimo_mes=tabela.index[-1], rmse_passeio_teste=round(rmse_passeio, 6),
)
print(sorted(p.name for p in PASTA.iterdir()))